# 02 — Silver Transformation

Reads new Bronze rows via **Change Data Feed** (incremental on subsequent runs,
full on first run), applies type casting and feature engineering, validates,
and writes to Silver tables.

## Key transformations
- `silver_race_results` — typed positions/points, status categorization, lap time parsing, gap-to-winner
- `silver_qualifying` — Q-time parsing to seconds, gap-to-pole, highest Q session reached
- `silver_driver_standings` — position_change via LAG window
- `silver_constructor_standings` — points_gap_to_leader via window
- `silver_lap_analysis` — join OpenF1 laps with Jolpica race schedule (via race_date) to add compound/tyre data

In [ ]:
%pip install /Workspace/Users/pravbala30@protonmail.com/.bundle/f1-intelligence/dev/artifacts/.internal/f1_intelligence-0.1.0-py3-none-any.whl --quiet
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("catalog", "f1_intelligence")
dbutils.widgets.text("schema",  "f1_dev")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")

print(f"catalog={CATALOG}  schema={SCHEMA}")

In [ ]:
# Setup
import re
import pyspark.sql.functions as F
from pyspark.sql import Window
from pyspark.sql.types import IntegerType, DoubleType, BooleanType, DateType

from utils.helpers import (
    get_or_create_catalog_schema,
    read_incremental_or_full,
    merge_delta,
    enable_cdf,
    save_checkpoint,
    get_current_table_version,
    add_metadata_columns,
    print_table_stats,
    table_exists,
)
from utils.schema import MERGE_KEYS
from utils.validators import validate_silver_results, validate_silver_laps

get_or_create_catalog_schema(spark, CATALOG, SCHEMA)

def T(table):  return f"{CATALOG}.{SCHEMA}.{table}"
CKPT = T("pipeline_checkpoints")

print("Setup complete")

In [ ]:
# ── Helper: parse lap time string '1:31.234' → seconds ─────────────────────
@F.udf(returnType=DoubleType())
def parse_lap_time(t):
    """Convert 'M:SS.mmm' or 'H:MM:SS.mmm' string to total seconds. Returns None on failure."""
    if not t:
        return None
    try:
        t = t.strip()
        parts = t.split(":")
        if len(parts) == 2:
            return float(parts[0]) * 60 + float(parts[1])
        elif len(parts) == 3:
            return float(parts[0]) * 3600 + float(parts[1]) * 60 + float(parts[2])
        return None
    except Exception:
        return None

@F.udf(returnType=DoubleType())
def parse_gap(gap_str):
    """Parse gap strings like '+5.123' or '+1 Lap' to seconds. Winner (no gap) → 0.0."""
    if not gap_str:
        return 0.0
    try:
        cleaned = gap_str.strip().lstrip("+")
        if "Lap" in cleaned:
            return None  # lapped cars — no meaningful seconds gap
        return float(cleaned)
    except Exception:
        return None

@F.udf(returnType=DoubleType())
def parse_race_time(time_str):
    """Parse race total time '1:32:05.123' → seconds. None for non-winners."""
    if not time_str:
        return None
    try:
        parts = time_str.strip().split(":")
        if len(parts) == 3:
            return float(parts[0]) * 3600 + float(parts[1]) * 60 + float(parts[2])
        elif len(parts) == 2:
            return float(parts[0]) * 60 + float(parts[1])
        return None
    except Exception:
        return None

@F.udf(returnType=DoubleType())
def safe_float(s):
    if not s:
        return None
    try:
        return float(s)
    except Exception:
        return None



def clean_int(col_expr):
    """Cast a StringType column to IntegerType, treating '' and literal 'None' as SQL NULL.
    Bronze data can contain the string 'None' when the API returned null and Python's
    str() was called on it (str(None) == 'None'). Direct .cast(IntegerType()) raises
    CAST_INVALID_INPUT on that literal, so we null-guard first.
    """
    return F.when(col_expr.isin("", "None"), F.lit(None)).otherwise(col_expr).cast(IntegerType())

STATUS_MAP = {
    "Finished": "FINISHED",
    "+1 Lap": "FINISHED", "+2 Laps": "FINISHED", "+3 Laps": "FINISHED",
}
MECHANICAL = {"Engine", "Gearbox", "Hydraulics", "Brakes", "Electrical",
              "Transmission", "Clutch", "Turbo", "Oil", "Water", "Fire",
              "Exhaust", "Wheel", "Suspension", "Differential", "Alternator",
              "Fuel", "ERS", "MGU", "Power Unit", "Overheating"}
ACCIDENT =   {"Accident", "Collision", "Spun off", "Damage"}

@F.udf(returnType=F.StringType() if hasattr(F, 'StringType') else None)
def categorize_status(status):
    if not status:
        return "OTHER"
    if status in ("Finished", "+1 Lap", "+2 Laps", "+3 Laps", "+4 Laps", "+5 Laps"):
        return "FINISHED"
    if status == "Disqualified":
        return "DISQUALIFIED"
    if status == "Did not qualify" or status == "Did not prequalify":
        return "DID_NOT_START"
    if any(m.lower() in status.lower() for m in MECHANICAL):
        return "RETIRED_MECHANICAL"
    if any(a.lower() in status.lower() for a in ACCIDENT):
        return "RETIRED_ACCIDENT"
    return "OTHER"

print("UDFs registered")


In [ ]:
# ── Silver Race Results ─────────────────────────────────────────────────────
print("Processing silver_race_results...")

df_bronze_results, bronze_results_version = read_incremental_or_full(
    spark, T("bronze_race_results"), CKPT, "bronze_to_silver_results"
)

if df_bronze_results.isEmpty():
    print("No new bronze_race_results — skipping silver_race_results")
else:
    # Join schedule for circuit_id and race_name
    df_schedule = spark.read.format("delta").table(T("bronze_race_schedule")) \
        .select("season", "round", "circuit_id", "race_name", "race_date")

    df_silver_results = (
        df_bronze_results
        .join(df_schedule, on=["season", "round"], how="left")
        .withColumn("season",               F.col("season").cast(IntegerType()))
        .withColumn("round",                F.col("round").cast(IntegerType()))
        .withColumn("race_date",            F.to_date("race_date", "yyyy-MM-dd"))
        .withColumn("driver_number",        F.col("driver_number").cast(IntegerType()))
        .withColumn("grid_position",        F.col("grid_position").cast(IntegerType()))
        .withColumn("final_position",       F.nullif(F.col("final_position"), F.lit("")).cast(IntegerType()))
        .withColumn("is_classified",        F.col("position_text").rlike(r"^\d+$"))
        .withColumn("points",               F.col("points").cast(DoubleType()))
        .withColumn("laps_completed",       F.col("laps_completed").cast(IntegerType()))
        .withColumn("race_time_seconds",    parse_race_time(F.col("race_time")))
        .withColumn("gap_to_winner_seconds",parse_gap(F.col("race_time")))
        .withColumn("fastest_lap_rank",     F.nullif(F.col("fastest_lap_rank"), F.lit("")).cast(IntegerType()))
        .withColumn("fastest_lap_seconds",  parse_lap_time(F.col("fastest_lap_time")))
        .withColumn("fastest_lap_speed_kph",safe_float(F.col("fastest_lap_speed")))
        .withColumn("status_category",      categorize_status(F.col("status")))
        .select(
            "season", "round", "circuit_id", "race_name", "race_date",
            "driver_id", "driver_code", "driver_number", "constructor_id",
            "grid_position", "final_position", "is_classified",
            "status", "status_category", "points", "laps_completed",
            "race_time_seconds", "gap_to_winner_seconds",
            "fastest_lap_rank", "fastest_lap_seconds", "fastest_lap_speed_kph",
            "ingested_at", "source_name",
        )
    )

    result = validate_silver_results(df_silver_results)
    result.raise_if_failed()

    merge_delta(spark, df_silver_results, T("silver_race_results"),
                MERGE_KEYS["silver_race_results"])
    print_table_stats(spark, T("silver_race_results"))
    print("silver_race_results done")

In [ ]:
# ── Silver Qualifying ───────────────────────────────────────────────────────
print("Processing silver_qualifying...")

df_bronze_qual = spark.read.format("delta").table(T("bronze_qualifying"))

if df_bronze_qual.isEmpty():
    print("No qualifying data — skipping (sprint weekends only?)")
else:
    df_schedule = spark.read.format("delta").table(T("bronze_race_schedule")) \
        .select("season", "round", "circuit_id", "race_date")

    pole_window = Window.partitionBy("season", "round")

    df_silver_qual = (
        df_bronze_qual
        .join(df_schedule, on=["season", "round"], how="left")
        .withColumn("season",               F.col("season").cast(IntegerType()))
        .withColumn("round",                F.col("round").cast(IntegerType()))
        .withColumn("race_date",            F.to_date("race_date", "yyyy-MM-dd"))
        .withColumn("qualifying_position",  F.col("qualifying_position").cast(IntegerType()))
        .withColumn("q1_seconds",           parse_lap_time(F.nullif(F.col("q1_time"), F.lit(""))))
        .withColumn("q2_seconds",           parse_lap_time(F.nullif(F.col("q2_time"), F.lit(""))))
        .withColumn("q3_seconds",           parse_lap_time(F.nullif(F.col("q3_time"), F.lit(""))))
        .withColumn("best_qualifying_time_seconds",
            F.coalesce(F.col("q3_seconds"), F.col("q2_seconds"), F.col("q1_seconds")))
        .withColumn("q_session_reached",
            F.when(F.col("q3_seconds").isNotNull(), F.lit(3))
             .when(F.col("q2_seconds").isNotNull(), F.lit(2))
             .otherwise(F.lit(1)))
        .withColumn("pole_time",
            F.min("best_qualifying_time_seconds").over(pole_window))
        .withColumn("gap_to_pole_seconds",
            F.when(F.col("qualifying_position") == 1, F.lit(0.0))
             .otherwise(F.col("best_qualifying_time_seconds") - F.col("pole_time")))
        .select(
            "season", "round", "circuit_id", "race_date",
            "driver_id", "driver_code", "constructor_id",
            "qualifying_position", "q1_seconds", "q2_seconds", "q3_seconds",
            "best_qualifying_time_seconds", "q_session_reached", "gap_to_pole_seconds",
            "ingested_at", "source_name",
        )
    )

    merge_delta(spark, df_silver_qual, T("silver_qualifying"),
                MERGE_KEYS["silver_qualifying"])
    print_table_stats(spark, T("silver_qualifying"))
    print("silver_qualifying done")

In [ ]:
# ── Silver Driver Standings ─────────────────────────────────────────────────
print("Processing silver_driver_standings...")

df_bronze_ds = spark.read.format("delta").table(T("bronze_driver_standings"))
lag_window   = Window.partitionBy("season", "driver_id").orderBy("round")

df_silver_ds = (
    df_bronze_ds
    .withColumn("season",          F.col("season").cast(IntegerType()))
    .withColumn("round",           F.col("round").cast(IntegerType()))
    .withColumn("position",        F.col("position").cast(IntegerType()))
    .withColumn("points",          F.col("points").cast(DoubleType()))
    .withColumn("wins",            F.col("wins").cast(IntegerType()))
    .withColumn("prev_position",   F.lag("position").over(lag_window))
    .withColumn("position_change",
        F.when(F.col("prev_position").isNotNull(),
               F.col("prev_position") - F.col("position"))  # positive = gained positions
         .otherwise(F.lit(0)))
    .select(
        "season", "round", "driver_id", "driver_code", "constructor_id",
        "position", "points", "wins", "position_change",
        "ingested_at", "source_name",
    )
)

merge_delta(spark, df_silver_ds, T("silver_driver_standings"),
            MERGE_KEYS["silver_driver_standings"])
print_table_stats(spark, T("silver_driver_standings"))
print("silver_driver_standings done")

In [ ]:
# ── Silver Constructor Standings ────────────────────────────────────────────
print("Processing silver_constructor_standings...")

df_bronze_cs = spark.read.format("delta").table(T("bronze_constructor_standings"))
leader_window = Window.partitionBy("season", "round")

df_silver_cs = (
    df_bronze_cs
    .withColumn("season",               F.col("season").cast(IntegerType()))
    .withColumn("round",                F.col("round").cast(IntegerType()))
    .withColumn("position",             F.col("position").cast(IntegerType()))
    .withColumn("points",               F.col("points").cast(DoubleType()))
    .withColumn("wins",                 F.col("wins").cast(IntegerType()))
    .withColumn("leader_points",        F.max("points").over(leader_window))
    .withColumn("points_gap_to_leader",
        F.col("leader_points") - F.col("points"))
    .select(
        "season", "round", "constructor_id", "constructor_name",
        "position", "points", "wins", "points_gap_to_leader",
        "ingested_at", "source_name",
    )
)

merge_delta(spark, df_silver_cs, T("silver_constructor_standings"),
            MERGE_KEYS["silver_constructor_standings"])
print_table_stats(spark, T("silver_constructor_standings"))
print("silver_constructor_standings done")

In [ ]:
# ── Silver Lap Analysis (OpenF1 × Jolpica join) ─────────────────────────────
print("Processing silver_lap_analysis...")

df_bronze_laps, bronze_laps_version = read_incremental_or_full(
    spark, T("bronze_laps"), CKPT, "bronze_to_silver_laps"
)

if df_bronze_laps.isEmpty():
    print("No new bronze_laps — skipping silver_lap_analysis")
    silver_laps_skipped = True
else:
    silver_laps_skipped = False
    df_bronze_stints = spark.read.format("delta").table(T("bronze_stints"))
    df_schedule      = spark.read.format("delta").table(T("bronze_race_schedule")) \
        .select("season", "round", "circuit_id", "race_date")
    # Alias season/round in results to avoid ambiguity after joining with df_session_map
    df_results       = spark.read.format("delta").table(T("bronze_race_results")) \
        .select(
            F.col("season").alias("result_season"),
            F.col("round").alias("result_round"),
            "driver_id", "driver_code", "constructor_id",
            F.col("driver_number").alias("driver_number_str"),
        )

    # Derive session date from the earliest lap date_start per session
    df_session_dates = (
        spark.read.format("delta").table(T("bronze_laps"))
        .withColumn("session_date", F.substring(F.col("date_start"), 1, 10))
        .groupBy("session_key")
        .agg(F.min("session_date").alias("session_date"))
    )

    # Map session_key → (season, round, circuit_id) by matching session date to race date
    df_session_map = (
        df_session_dates
        .join(df_schedule, df_session_dates["session_date"] == df_schedule["race_date"], "left")
        .select("session_key", "season", "round", "circuit_id")
    )

    # Join stints to laps on (session_key, driver_number, lap_number within stint range)
    df_stints_typed = (
        df_bronze_stints
        .withColumn("stint_number",  clean_int(F.col("stint_number")))
        .withColumn("lap_start",     clean_int(F.col("lap_start")))
        .withColumn("lap_end",       clean_int(F.col("lap_end")))
        .select("session_key", "driver_number", "stint_number", "lap_start", "lap_end",
                "compound", "tyre_age_at_start")
    )

    df_laps_typed = (
        df_bronze_laps
        .withColumn("lap_number", clean_int(F.col("lap_number")))
    )

    # Each lap gets the stint where lap_start <= lap_number <= lap_end
    df_laps_stints = (
        df_laps_typed.alias("l")
        .join(
            df_stints_typed.alias("s"),
            (
                (F.col("l.session_key")   == F.col("s.session_key")) &
                (F.col("l.driver_number") == F.col("s.driver_number")) &
                (F.col("l.lap_number")    >= F.col("s.lap_start")) &
                (F.col("l.lap_number")    <= F.col("s.lap_end"))
            ),
            how="left",
        )
        .select(
            F.col("l.session_key"),
            F.col("l.driver_number"),
            F.col("l.lap_number"),
            safe_float(F.col("l.lap_duration")).alias("lap_duration_seconds"),
            F.when(F.col("l.is_pit_out_lap") == "true", True).otherwise(False).alias("is_pit_out_lap"),
            safe_float(F.col("l.duration_sector_1")).alias("sector_1_seconds"),
            safe_float(F.col("l.duration_sector_2")).alias("sector_2_seconds"),
            safe_float(F.col("l.duration_sector_3")).alias("sector_3_seconds"),
            safe_float(F.col("l.i1_speed")).alias("i1_speed_kph"),
            safe_float(F.col("l.i2_speed")).alias("i2_speed_kph"),
            safe_float(F.col("l.st_speed")).alias("st_speed_kph"),
            F.col("s.stint_number"),
            F.col("s.compound"),
            clean_int(F.col("s.tyre_age_at_start")).alias("tyre_age_at_start"),
            F.col("l.ingested_at"),
            F.col("l.source_name"),
        )
    )

    # Add tyre_age_laps = tyre_age_at_start + (lap_number - lap_start)
    df_laps_stints = (
        df_laps_stints
        .join(
            df_stints_typed.select("session_key", "driver_number", "stint_number", "lap_start"),
            on=["session_key", "driver_number", "stint_number"],
            how="left",
        )
        .withColumn("tyre_age_laps",
            F.when(F.col("tyre_age_at_start").isNotNull() & F.col("lap_start").isNotNull(),
                   F.col("tyre_age_at_start") + (F.col("lap_number") - F.col("lap_start")))
             .otherwise(F.lit(None).cast(IntegerType())))
        .drop("tyre_age_at_start", "lap_start")
    )

    # Personal best per driver per session
    pb_window = Window.partitionBy("session_key", "driver_number")
    df_laps_stints = df_laps_stints.withColumn(
        "min_lap_duration", F.min("lap_duration_seconds").over(pb_window)
    ).withColumn(
        "is_personal_best",
        (F.col("lap_duration_seconds") == F.col("min_lap_duration")) &
        F.col("lap_duration_seconds").isNotNull()
    ).drop("min_lap_duration")

    # Join session map to get season/round/circuit_id
    df_with_session = df_laps_stints.join(df_session_map, on="session_key", how="left")

    # Join race results for driver_id and team — use aliased season/round to avoid ambiguity
    df_final_laps = (
        df_with_session
        .join(
            df_results,
            (
                (F.col("season") == df_results["result_season"]) &
                (F.col("round")  == df_results["result_round"]) &
                (F.col("driver_number") == df_results["driver_number_str"])
            ),
            how="left",
        )
        .drop("result_season", "result_round", "driver_number_str")
        .withColumn("season",        clean_int(F.col("season")))
        .withColumn("round",         clean_int(F.col("round")))
        .withColumn("driver_number", clean_int(F.col("driver_number")))
        .select(
            "session_key", "season", "round", "circuit_id",
            "driver_number", "driver_id", "driver_code", "constructor_id",
            "lap_number", "lap_duration_seconds", "is_pit_out_lap",
            "sector_1_seconds", "sector_2_seconds", "sector_3_seconds",
            "i1_speed_kph", "i2_speed_kph", "st_speed_kph",
            "stint_number", "compound", "tyre_age_laps", "is_personal_best",
            "ingested_at", "source_name",
        )
    )

    result = validate_silver_laps(df_final_laps)
    result.raise_if_failed()

    merge_delta(spark, df_final_laps, T("silver_lap_analysis"),
                MERGE_KEYS["silver_lap_analysis"])
    print_table_stats(spark, T("silver_lap_analysis"))
    print("silver_lap_analysis done")



In [ ]:
# ── Save checkpoints ────────────────────────────────────────────────────────
if not df_bronze_results.isEmpty():
    save_checkpoint(
        spark, CKPT, "bronze_to_silver_results",
        processed_version=bronze_results_version,
        records_processed=df_silver_results.count(),
    )

if not df_bronze_laps.isEmpty() and not silver_laps_skipped:
    save_checkpoint(
        spark, CKPT, "bronze_to_silver_laps",
        processed_version=bronze_laps_version,
        records_processed=df_final_laps.count(),
    )

print("\n=== Silver transformation complete ===")
for t in ["silver_race_results", "silver_qualifying", "silver_driver_standings",
           "silver_constructor_standings", "silver_lap_analysis"]:
    if table_exists(spark, T(t)):
        print_table_stats(spark, T(t))

dbutils.notebook.exit("silver_transformation_complete")